# Setup

In [ ]:
import sys
import json
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.py").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from config import MODEL_NAME, BASE_MODEL_URL, API_KEY_ENV, TRINO_CATALOG
from lakehouse.render import render_collective
from lakehouse.lakehouse import Lakehouse
from diversql.baselines import sqlstorm, sqlbarber, e2etune, bootstrapping_lcm, sql_factory
from diversql.baselines.common import context_from_lakehouse

INSTANCE_NAME = "lakehouse-a"
NAMESPACE = ""
SCHEMA = "tpcds"
CATALOG = TRINO_CATALOG

# Deploy Lakehouse

In [ ]:
"""
Deploy lakehouse with the tpcds schema and register to HMS
"""

res = render_collective(
    templates_dir=str(ROOT / "lakehouse" / "manifests"),
    values_path=str(ROOT / "lakehouse" / "lakehouses" / "lakehouse-tpcds.yaml"),
)

lh = Lakehouse(res, verbose=False)

lh.deploy()
lh.register_schema(SCHEMA, "/mnt/iceberg/warehouse", use_dot_db=True)  # <schema name in /warehouse>  <location of /warehouse in control container>


# Generate Queries

In [ ]:
import logging; logging.getLogger("httpx").setLevel(logging.WARNING)

In [ ]:

"""
Generate a query workload using the OpenAI API 
"""
lh = Lakehouse.from_release(instance_name=INSTANCE_NAME, namespace=NAMESPACE, sync_schemas=True, verbose=True)


report = lh.generate_workload(
    schema=SCHEMA,
    workload_name="diverSQL",
    num_queries=1000,
    min_tables=2,
    max_tables=16,
    warmup=True,
    validate=True,
    batch_size=100,
    generation_workers=64,
    validation_workers=8,
    reasoning="high",
    request_timeout_s=1200,
    plan_feedback=False,
)


# Config

In [ ]:
"""
Hardcoded config used by every generation cell below.
Edit these values to change the model settings and volume (schema and instance are set in Setup).
"""

NUM_QUERIES = 1000

TEMPERATURE = 0.6
REASONING = "high"

RANDOM_SEED = 42
WARMUP = True

ctx = context_from_lakehouse(
    lh,
    schema=SCHEMA,
    catalog=CATALOG,
    model_name=MODEL_NAME,
    base_url=BASE_MODEL_URL,
    api_key_env=API_KEY_ENV,
    temperature=TEMPERATURE,
    reasoning=REASONING,
)


# Generate Baseline Workloads

## sqlstorm

In [ ]:
"""
Generate a workload using the sqlstorm baseline
"""

WORKLOAD_NAME_SQLSTORM = "sqlstorm_1000_tokens_{}".format(SCHEMA)

report_sqlstorm = sqlstorm.generate_workload(
    ctx,
    workload_name=WORKLOAD_NAME_SQLSTORM,
    num_queries=NUM_QUERIES,
    warmup=WARMUP,
    random_seed=RANDOM_SEED,
)

print(json.dumps({
    "baseline": report_sqlstorm["baseline"],
    "workload_dir": report_sqlstorm["workload_dir"],
    "num_queries": report_sqlstorm["num_queries"],
    "duration_s": round(report_sqlstorm["duration_s"], 1),
}, indent=2, default=str))


## sqlbarber

In [ ]:
"""
Generate a workload using the sqlbarber baseline
"""

WORKLOAD_NAME_SQLBARBER = "sqlbarber_1000_tokens_new_{}".format(SCHEMA)

report_sqlbarber = sqlbarber.generate_workload(
    ctx,
    workload_name=WORKLOAD_NAME_SQLBARBER,
    num_queries=NUM_QUERIES,
    warmup=WARMUP,
    random_seed=RANDOM_SEED,
)

print(json.dumps({
    "baseline": report_sqlbarber["baseline"],
    "workload_dir": report_sqlbarber["workload_dir"],
    "num_queries": report_sqlbarber["num_queries"],
    "duration_s": round(report_sqlbarber["duration_s"], 1),
}, indent=2, default=str))


## e2etune

In [ ]:
"""
Generate a workload using the e2etune baseline
"""

WORKLOAD_NAME_E2ETUNE = "e2etune_1000_tokens_{}".format(SCHEMA)

report_e2etune = e2etune.generate_workload(
    ctx,
    workload_name=WORKLOAD_NAME_E2ETUNE,
    num_queries=NUM_QUERIES,
    warmup=WARMUP,
    random_seed=RANDOM_SEED,
)

print(json.dumps({
    "baseline": report_e2etune["baseline"],
    "workload_dir": report_e2etune["workload_dir"],
    "num_queries": report_e2etune["num_queries"],
    "duration_s": round(report_e2etune["duration_s"], 1),
}, indent=2, default=str))


## bootstrapping_lcm

In [ ]:
"""
Generate a workload using the bootstrapping_lcm baseline
"""

WORKLOAD_NAME_BOOTSTRAPPING_LCM = "bootstrapping_lcm_1000_tokens_new_{}".format(SCHEMA)

report_bootstrapping_lcm = bootstrapping_lcm.generate_workload(
    ctx,
    workload_name=WORKLOAD_NAME_BOOTSTRAPPING_LCM,
    num_queries=NUM_QUERIES,
    warmup=WARMUP,
    random_seed=RANDOM_SEED,
)

print(json.dumps({
    "baseline": report_bootstrapping_lcm["baseline"],
    "workload_dir": report_bootstrapping_lcm["workload_dir"],
    "num_queries": report_bootstrapping_lcm["num_queries"],
    "duration_s": round(report_bootstrapping_lcm["duration_s"], 1),
}, indent=2, default=str))


## sql_factory

In [ ]:
"""
Generate a workload using the sql_factory baseline.

"""

WORKLOAD_NAME_SQL_FACTORY = "sql_factory_1000_tokens_new_{}".format(SCHEMA)

report_sql_factory = sql_factory.generate_workload(
    ctx,
    workload_name=WORKLOAD_NAME_SQL_FACTORY,
    num_queries=NUM_QUERIES,
    max_stall_rounds=10,
    warmup=WARMUP,
    random_seed=RANDOM_SEED,
)

print(json.dumps({
    "baseline": report_sql_factory["baseline"],
    "workload_dir": report_sql_factory["workload_dir"],
    "num_queries": report_sql_factory["num_queries"],
    "duration_s": round(report_sql_factory["duration_s"], 1),
    "stop_reason": report_sql_factory["pipeline"]["termination"]["stop_reason"],
}, indent=2, default=str))


# Tear Down Lakehouse

In [ ]:
"""
Tear down the lakehouse deployed in the first cell
"""

lh.tear_down()
